#### 1. Implement Heap

In [11]:
from heapq import heapify, heappush, heappop
from typing import *

##### 1. Kth Largest Element in a Stream

In [12]:
class KthLargest:
    def __init__(self, k: int, nums: List[int]):
        # always store top k scores in min heap
        self.min_heap = nums
        self.k = k
        heapify(self.min_heap)
        while len(self.min_heap) > k:
            heappop(self.min_heap)
                         
    def add(self, val: int) -> int:
        heapq.heappush(self.min_heap, val)
        while len(self.min_heap) > self.k:
            heappop(self.min_heap)
        return self.min_heap[0]

##### 2. Last Stone Weight

In [15]:
def lastStoneWeight(self, stones: List[int]) -> int:
    max_heap = []
    heapify(max_heap)
    for s in stones:
        heapq.heappush(max_heap, -s)
    
    while len(max_heap) > 1:
        a = -heappop(max_heap)
        b = -heappop(max_heap)
        if a != b:
            heappush(max_heap, -(a-b))
    # edge case
    return -heappop(max_heap) if len(max_heap) else 0

##### 3. Take Gifts From the Richest Pile

In [16]:
class Solution:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        max_heap = []
        heapify(max_heap)
        for g in gifts:
            heappush(max_heap, -g)
        for i in range(k):
            num = heappop(max_heap)
            heappush(max_heap, -math.floor(math.sqrt(-num)))
        ans = 0
        while len(max_heap):
            ans += heappop(max_heap)
        return -ans

##### 4.Final Array State After K Multiplication Operations I

In [18]:
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        min_heap = [(num, idx) for idx,num in enumerate(nums)]
        heapify(min_heap)
        for i in range(k):
            num, idx = heappop(min_heap)
            nums[idx] = num * multiplier
            heappush(min_heap, (nums[idx],idx))
        return nums

##### 5. K Closest Points to Origin

In [19]:
class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        min_heap = []
        heapify(min_heap)
        for i in range(len(points)):
            x,y = points[i]
            heappush(min_heap, (math.sqrt(x**2 + y**2),i))
        ans = []
        for i in range(k):
            _,idx = heappop(min_heap)
            ans.append(points[idx])
        return ans

##### 6. Kth Largest Element

In [20]:
class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        # store top k largest elements
        max_heap = []
        heapify(max_heap)
        for num in nums:
            heappush(max_heap, num)
        while len(max_heap) > k:
            heappop(max_heap)
        return max_heap[0]

### 7. Task Scheduler - OG

In [21]:
class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        counts = Counter(tasks)
        # process most frequent ones
        max_heap = [-cnt for cnt in counts.values()]
        heapify(max_heap)
        q = deque()    # pair of (cnt,nextAvailbleTime)
        time = 0
        while max_heap or q:
            time += 1
            if max_heap:
                cnt = -heappop(max_heap) - 1
                if cnt:
                    q.append((-cnt, time + n))
            if q and q[0][1] == time:
                heappush(max_heap, q.popleft()[0])
        return time

### 8. Design Twitter - OG

In [22]:
class Twitter:

    def __init__(self):
        self.time = 0
        self.followers_map = defaultdict(set) # followers_id
        self.tweet_map = defaultdict(list) # (time, tweet_id)
    
    def postTweet(self, userId: int, tweetId: int) -> None:
        self.tweet_map[userId].append((self.time, tweetId))
        self.time += 1

    def getNewsFeed(self, userId: int) -> List[int]:
        ans = []
        max_heap = []
        # edge case
        self.followers_map[userId].add(userId)
        for followee in self.followers_map[userId]:
            if followee in self.tweet_map:
                last_idx = len(self.tweet_map[followee]) - 1
                time, tweet_id = self.tweet_map[followee][last_idx]
                next_idx = last_idx - 1
                max_heap.append([-time, tweet_id, next_idx, followee])
        heapify(max_heap)
        while max_heap and len(ans) < 10:
            time, tweet_id, idx, followee = heappop(max_heap)
            ans.append(tweet_id)
            if idx >= 0:
                time, tweet_id = self.tweet_map[followee][idx]
                next_idx = idx - 1
                heappush(max_heap, [-time, tweet_id, next_idx, followee])
        return ans

    def follow(self, followerId: int, followeeId: int) -> None:
        self.followers_map[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        if followeeId in self.followers_map[followerId]:
            self.followers_map[followerId].remove(followeeId)

##### 9. Furthest Building You Can Reach - Good

In [23]:
class Solution:
    def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int:
        max_heap = []   # used bricks

        for i in range(len(heights)-1):
            diff = heights[i+1] - heights[i]
            if diff <= 0:
                continue
            
            # use bricks frist
            bricks -= diff
            heappush(max_heap, -diff)
            if bricks < 0:
                if ladders == 0:
                    return i
                # use ladder before and get the bricks
                bricks += -heappop(max_heap)
                ladders -= 1
        return len(heights) - 1

##### 10. Maximum Subsequence Score - Good

In [24]:
class Solution:
    def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
        # Intution - (n1[0], n1[1], n1[2]..) * min(n2[0],n2[1],n2[2],..)
        # keep track of max in n2
        # and remove min from n2 always if overgrows
        pairs = [(x,y) for x,y in zip(nums1,nums2)]
        pairs.sort(key=lambda x:x[1], reverse=True)
        
        min_heap = []
        n1_sum = 0
        ans = 0
        for x,y in pairs:
            n1_sum += x
            heappush(min_heap, x)

            if len(min_heap) > k:
                # pop min of nums1
                n1_min = heappop(min_heap)
                n1_sum -= n1_min
            if len(min_heap) == k:
                ans = max(ans, n1_sum * y)
        return ans

### 11 - Single Threaded CPU - Good

In [25]:
class Solution:
    def getOrder(self, tasks: List[List[int]]) -> List[int]:
        n = len(tasks)
        pairs = []
        for i in range(n):
            pairs.append((tasks[i][0],tasks[i][1],i))
        pairs.sort()
        time = 0
        i = 0
        min_heap = []
        ans = []
        while i < n or min_heap:
            if not min_heap and time < pairs[i][0]:
                time = pairs[i][0]
            
            while i < n and time >= pairs[i][0]:
                s,e,idx = pairs[i]
                heappush(min_heap, (e,idx))
                i += 1
            e,idx = heappop(min_heap)
            time += e
            ans.append(idx)
        return ans


### 12. Process Tasks Using Servers - OG

In [26]:
class Solution:
    def assignTasks(self, servers: List[int], tasks: List[int]) -> List[int]:
        n, m = len(servers), len(tasks)
        available_servers = [(wgt,idx) for idx,wgt in enumerate(servers)]
        heapify(available_servers)
        unavailable_servers = []
        task_pairs = [(s,e) for s,e in enumerate(tasks)]
        time = 0
        ans = []
        for s,e in task_pairs:
            time = max(time, s)

            # if all are unavailable
            if len(available_servers) == 0:
                time = unavailable_servers[0][0]
            
            # check if any unavailable servers got freed
            while unavailable_servers and time >= unavailable_servers[0][0]:
                _,wgt,idx = heappop(unavailable_servers)
                heappush(available_servers, (wgt,idx))
            
            # assign server to task
            wgt,idx = heappop(available_servers)
            ans.append(idx)
            heappush(unavailable_servers, (time+e,wgt,idx))
        
        return ans

### 13. Reorganize String - OG

In [27]:
class Solution:
    def reorganizeString(self, s: str) -> str:
        counts = Counter(s)
        max_heap = [(-cnt,char) for char,cnt in counts.items()]
        heapify(max_heap)

        ans = ""

        while max_heap:
            # get the most frequent char
            cnt, char = heappop(max_heap)
            if len(ans) and ans[-1] == char:
                # you shouldn't take this character
                # note: if the max_heap is empty, that means not possible
                if not max_heap:
                    return ""
                new_cnt, new_char = heappop(max_heap)
                ans += new_char
                if new_cnt + 1 < 0:
                    heappush(max_heap, (new_cnt+1,new_char))
                # add back the char
                heappush(max_heap, (cnt,char))
            else:
                # can add this to the result
                ans += char
                if cnt + 1 < 0:
                    heappush(max_heap, (cnt+1,char))
        
        return ans

### 14. Longest Happy String

In [29]:
class Solution:
    def longestDiverseString(self, a: int, b: int, c: int) -> str:
        max_heap = []
        if a > 0: heappush(max_heap, (-a,'a'))
        if b > 0: heappush(max_heap, (-b,'b'))
        if c > 0: heappush(max_heap, (-c,'c'))
        heapify(max_heap)
        ans = ""
        while max_heap:
            cnt, char = heappop(max_heap)
            
            if len(ans) >= 2:
                if (ans[-2:] == 'aa' and char == 'a') or \
                    (ans[-2:] == 'bb' and char == 'b') or \
                    (ans[-2:] == 'cc' and char == 'c'):
                    # get another character
                    if not max_heap:
                        break
                    new_cnt, new_char = heappop(max_heap)
                    ans += new_char
                    if new_cnt + 1 < 0:
                        heappush(max_heap, (new_cnt+1,new_char))
                    # push back the original one
                    heappush(max_heap, (cnt,char))
                else:
                    # can add anything here 
                    ans += char
                    if cnt + 1 < 0:
                        heappush(max_heap, (cnt+1,char))
            else:
                ans += char
                if cnt + 1 < 0:
                    heappush(max_heap,(cnt+1,char))
        return ans